# Importing Libraries

In [33]:
import pandas as pd
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

In [34]:
base_dir = Path.cwd()
data_path = (base_dir/'Data'/'co2_emissions.csv')
a = pd.read_csv(data_path)

In [35]:
a.columns

Index(['Country', 'Region', 'Year', 'CO2_Mt', 'CO2_per_capita'], dtype='str')

In [36]:
a.head()

,Country,Region,Year,CO2_Mt,CO2_per_capita
0,United States,North America,2000,5857.6,1.32
1,United States,North America,2001,5724.0,1.26
2,United States,North America,2002,5652.8,1.11
3,United States,North America,2003,5592.8,1.29
4,United States,North America,2004,5743.2,1.12


In [37]:
a.info()

<class 'pandas.DataFrame'>
RangeIndex: 345 entries, 0 to 344
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country         345 non-null    str    
 1   Region          345 non-null    str    
 2   Year            345 non-null    int64  
 3   CO2_Mt          345 non-null    float64
 4   CO2_per_capita  345 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 13.6 KB


In [38]:
a.describe()

,Year,CO2_Mt,CO2_per_capita
count,345.000000,345.000000,345.000000
mean,2011.000000,1485.344638,1.478435
std,6.642884,2199.663781,0.695194
min,2000.000000,125.300000,0.310000
25%,2005.000000,458.700000,1.080000
50%,2011.000000,570.000000,1.300000
75%,2017.000000,1291.000000,1.640000
max,2022.000000,12409.500000,4.370000


In [39]:
print("Countries:", a['Country'].unique())
print("\nCO2 range:", a['CO2_Mt'].min(), "to", a['CO2_Mt'].max(), "Mt")
print(a[a['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


In [40]:
asia_df = a[a['Region'] == 'Asia']
highlight = 'India'

fig = go.Figure()

for country in asia_df['Country'].unique():
    country_data = asia_df[asia_df['Country'] == country].sort_values('Year')
    
    if country == highlight:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#FF4500', width=3),  # thick orange
            showlegend=False
        ))
    else:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#DDDDDD', width=1),  # thin grey
            showlegend=False,
            opacity=0.4
        ))


last = asia_df[asia_df['Country'] == highlight].sort_values('Year').iloc[-1]
fig.add_annotation(
    x=last['Year'],
    y=last['CO2_Mt'],
    text=f"<b>{highlight}</b>",
    showarrow=False,
    xanchor='left',
    font=dict(color='#FF4500', size=13)
)

fig.update_layout(
    title="India is Rising: CO2 Emissions Climbing Fast While Asia Watches",
    title_font=dict(size=18, color='#FF4500'),
    paper_bgcolor='#1E1E2E',
    plot_bgcolor='#2A2A3E',
    font=dict(color='white'),
    xaxis=dict(title='Year', showgrid=False),
    yaxis=dict(title='CO2 Emissions (Mt)', showgrid=True, gridcolor='#444444'),
    width=1150, height=580
)

fig.show()

In [43]:
import plotly.graph_objects as go

regional = a.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()

r2000 = regional[regional['Year'] == 2000].set_index('Region')['CO2_Mt']
r2022 = regional[regional['Year'] == 2022].set_index('Region')['CO2_Mt']

regions = r2000.index

fig = go.Figure()


used_y_left = []
used_y_right = []

def adjust_y(y, used, min_gap=120):
    for taken in used:
        if abs(y - taken) < min_gap:
            y = taken + min_gap
    used.append(y)
    return y

for region in regions:
    val2000 = r2000[region]
    val2022 = r2022[region]
    increased = val2022 > val2000
    color = '#FF4500' if increased else '#4499FF'

    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[val2000, val2022],
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=8, color=color),
        showlegend=False
    ))

    y_left = adjust_y(val2000, used_y_left)
    y_right = adjust_y(val2022, used_y_right)

    fig.add_annotation(
        x=2000, y=y_left,
        text=f"{region}: {val2000:.0f}",
        xanchor='right', xshift=-10,
        showarrow=False,
        font=dict(color=color, size=11)
    )

    fig.add_annotation(
        x=2022, y=y_right,
        text=f"{val2022:.0f}",
        xanchor='left', xshift=10,
        showarrow=False,
        font=dict(color=color, size=11)
    )

fig.update_layout(
    title="Asia & Middle East Surged Most — Regional CO2 Shift 2000 vs 2022",
    title_font=dict(size=18, color='white'),
    paper_bgcolor='#1E1E2E',
    plot_bgcolor='#2A2A3E',
    font=dict(color='white'),
    xaxis=dict(
        tickvals=[2000, 2022],
        showgrid=False
    ),
    yaxis=dict(
        showticklabels=False,
        showgrid=False
    ),
    width=1100, height=650
)

fig.show()